# Naive Bayes Classification in Python

Dataset from Kaggle : **"Netflix Movies and TV Shows"** by *Shivam Bansal*  
Source: https://www.kaggle.com/datasets/shivamb/netflix-shows (requires login)

---

### Essential Libraries

Let us begin by importing the essential Python Libraries.

> NumPy : Library for Numeric Computations in Python  
> Pandas : Library for Data Acquisition and Preparation  
> Matplotlib : Low-level library for Data Visualization  
> Seaborn : Higher-level library for Data Visualization  

In [ ]:
# Basic Libraries
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt # we only need pyplot
sb.set() # set the default Seaborn style for graphics

In [ ]:
# Print library package version
print("numpy version = "+np.__version__)
print("pandas version = "+pd.__version__)
print("seaborn version = "+sb.__version__)

---

### Import the Dataset

The dataset is in CSV format; hence we use the `read_csv` function from Pandas.  
Immediately after importing, take a quick look at the data using the `head` function.

In [ ]:
netflixdata = pd.read_csv('netflix_titles.csv')
netflixdata.head()

Description of the dataset, as available on Kaggle, is as follows.

> **show_id** : Unique ID for each title  
> **type** : Whether the title is a Movie or a TV Show  
> **title** : Name of the title  
> **director** : Director(s) of the title  
> **cast** : Main actors in the title  
> **country** : Country or countries where the title was produced  
> **date_added** : Date the title was added to Netflix  
> **release_year** : Year the title was originally released  
> **rating** : Age rating of the title (e.g. TV-MA, PG-13)  
> **duration** : Minutes for movies, number of seasons for TV shows  
> **listed_in** : Genres of the title, separated by commas  
> **description** : Short summary of the title  

---

Check the vital statistics of the dataset using the `type` and `shape` attributes.

In [ ]:
print("Data type : ", type(netflixdata))
print("Data dims : ", netflixdata.shape)

Check the variables (and their types) in the dataset using the `dtypes` attribute.

In [ ]:
print(netflixdata.dtypes)

---

## Clean the Dataset

Before setting up the problem, check the dataset for missing values, duplicates and invalid entries.

In [ ]:
# Information about the Variables
netflixdata.info()

In [ ]:
# Missing values in each Variable
print(netflixdata.isnull().sum())

In [ ]:
# Duplicate rows and duplicate descriptions
print("Duplicate rows :", netflixdata.duplicated().sum())
print("Duplicate descriptions :", netflixdata["description"].duplicated().sum())

#### Ratings in the Dataset

Some entries in **rating** are not valid age ratings -- *NR* and *UR* (not rated), and a few durations such as *74 min* entered in the wrong column.

In [ ]:
# Ratings in the Dataset
print("Number of Ratings :", len(netflixdata["rating"].dropna().unique()))

# Titles of each Rating
print(netflixdata["rating"].value_counts(dropna = False))
sb.catplot(y = "rating", data = netflixdata, kind = "count",
           order = netflixdata["rating"].value_counts().index, height = 8)

#### Remove Unusable Titles

Keep only titles with a valid age rating, a description and a genre, and remove repeated descriptions.

In [ ]:
# Valid Age Ratings
valid_ratings = ["TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "G", "PG", "TV-PG",
                 "PG-13", "TV-14", "R", "TV-MA", "NC-17"]

# Keep only the usable Titles
netflixclean = netflixdata[netflixdata["rating"].isin(valid_ratings)]
netflixclean = netflixclean.dropna(subset = ["description", "listed_in"])
netflixclean = netflixclean.drop_duplicates(subset = "description")

# Check the sample sizes
print("Before Cleaning :", netflixdata.shape)
print("After Cleaning  :", netflixclean.shape)

---

## Set up the Classification Problem

We will set up a Classification Problem as follows.   

Response Variable : **Adult** (whether the title is rated 18+)     
Predictor Features : **description** and **listed_in** (genre)    

A title is **18+** if its rating is *R*, *TV-MA* or *NC-17*, and **Not 18+** otherwise.

In [ ]:
# Create the Response Variable
netflixclean["adult"] = netflixclean["rating"].isin(["R", "TV-MA", "NC-17"])
netflixclean["adult"] = netflixclean["adult"].map({True : "18+", False : "Not 18+"})

# Titles in each Class
print(netflixclean["adult"].value_counts())
sb.catplot(y = "adult", data = netflixclean, kind = "count")

#### Combine Description and Genre

Each genre is converted into a single token with a `g_` prefix -- e.g. *International Movies* becomes `g_International_Movies`.  
This keeps a multi-word genre as one feature, and keeps genres separate from the words in the description.

In [ ]:
# Convert the Genres of each title into Genre tokens
def genre_tokens(genres):
    tokens = []
    for g in genres.split(", "):
        g = g.replace(" ", "_").replace("'", "").replace("-", "_").replace("&", "and")
        tokens.append("g_" + g)
    return " ".join(tokens)

netflixclean["genre"] = netflixclean["listed_in"].apply(genre_tokens)

# Combine Description and Genre into a single text
netflixclean["text"] = netflixclean["description"] + " " + netflixclean["genre"]

# Check the first title
print(netflixclean["listed_in"].iloc[0])
print(netflixclean["text"].iloc[0])

---

## Explore the Dataset

Perform basic statistical exploration and visualization on the Response and Predictors.

#### Baseline Accuracy

If we always guess the most common class, this is the accuracy we would get. Any model should beat this.

In [ ]:
# Proportion of each Class
print(netflixclean["adult"].value_counts(normalize = True))
print()
print("Baseline Accuracy \t:", netflixclean["adult"].value_counts(normalize = True).max())

#### Length of the Description

In [ ]:
# Number of words in each Description
desc_len = pd.DataFrame(netflixclean["description"].str.split().str.len())
desc_len.columns = ["Length"]

# Summary Statistics for Description Length
desc_len.describe()

In [ ]:
# Plot the distributions of Description Length
f, axes = plt.subplots(1, 3, figsize=(24, 6))
sb.boxplot(data = desc_len, orient = "h", ax = axes[0])
sb.histplot(data = desc_len, ax = axes[1])
sb.violinplot(data = desc_len, orient = "h", ax = axes[2])

In [ ]:
# Create a joint dataframe by concatenating Length and Adult
lenDF = pd.concat([desc_len, netflixclean["adult"]], axis = 1).reindex(desc_len.index)

# Joint Boxplot of Length against Adult
f = plt.figure(figsize=(18, 6))
sb.boxplot(x = "Length", y = "adult", data = lenDF, orient = "h")

#### Most Common Words in each Class

In [ ]:
# Import CountVectorizer from sklearn.feature_extraction.text
from sklearn.feature_extraction.text import CountVectorizer

# Top 15 words (ignoring common English words) in each Class
for cls in ["18+", "Not 18+"]:
    words = CountVectorizer(stop_words = "english")
    counts = words.fit_transform(netflixclean[netflixclean["adult"] == cls]["description"])
    top = pd.Series(counts.sum(axis = 0).A1, index = words.get_feature_names_out())
    print(cls, ":", list(top.nlargest(15).index))
    print()

#### Genres against Adult

Share of 18+ titles within each genre (genres with at least 100 titles).

In [ ]:
# One row for each Genre of each title
genreDF = netflixclean[["listed_in", "adult"]].copy()
genreDF["listed_in"] = genreDF["listed_in"].str.split(", ")
genreDF = genreDF.explode("listed_in").reset_index(drop = True)

# Count the titles in each Genre and Class
genre_counts = genreDF.groupby(["listed_in", "adult"]).size().unstack(fill_value = 0)
genre_counts = genre_counts[genre_counts.sum(axis = 1) >= 100]

# Share of 18+ titles in each Genre
genre_share = genre_counts["18+"] / genre_counts.sum(axis = 1)
genre_share = genre_share.sort_values()
print(genre_share.round(2))

In [ ]:
# Plot the Share of 18+ titles in each Genre
f = plt.figure(figsize=(12, 12))
sb.barplot(x = genre_share.values, y = genre_share.index, orient = "h")
plt.axvline(0.5, color = "black", linestyle = "--")
plt.xlabel("Share of 18+ titles")

---

## Naive Bayes Classifier

#### Split into Train and Test

Split the dataset randomly into Train (80%) and Test (20%) sets.  
`stratify` keeps the same proportion of 18+ and Not 18+ titles in both sets.

In [ ]:
# Import train_test_split from sklearn
from sklearn.model_selection import train_test_split

# Extract Response and Predictors
y = pd.DataFrame(netflixclean["adult"])   # Response
X = pd.DataFrame(netflixclean["text"])    # Predictor

# Split the Dataset into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,
                                                    stratify = y, random_state = 42)

# Check the sample sizes
print("Train Set :", y_train.shape, X_train.shape)
print("Test Set  :", y_test.shape, X_test.shape)

#### Convert Text to Word Counts

A model can only work with numbers, so each title's text is converted into word counts using `CountVectorizer`.  
The vocabulary is learnt from the **Train Set only**, then the same vocabulary is used on the Test Set.

In [ ]:
# Create a CountVectorizer object (keep every word, including genre tokens)
vectorizer = CountVectorizer(token_pattern = r"(?u)\b\w+\b")

# Learn the vocabulary from Train, and count words in Train and Test
X_train_counts = vectorizer.fit_transform(X_train["text"])
X_test_counts = vectorizer.transform(X_test["text"])

# Check the sizes (titles, words in vocabulary)
print("Train Counts :", X_train_counts.shape)
print("Test Counts  :", X_test_counts.shape)

#### Choose the Smoothing Parameter

`alpha` is the smoothing in Naive Bayes -- it adds a small count to every word so no word has a probability of zero.  
We pick the best `alpha` using 5-fold Cross-Validation on the Train Set.

In [ ]:
# Import MultinomialNB and cross_val_score from sklearn
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

# Try different values of alpha
for alpha in [0.1, 0.5, 1, 2]:
    nbmodel = MultinomialNB(alpha = alpha)
    scores = cross_val_score(nbmodel, X_train_counts, y_train["adult"], cv = 5, scoring = "f1_macro")
    print("alpha =", alpha, "\tMacro-F1 :", round(scores.mean(), 3))

#### Naive Bayes in Scikit-Learn

Train the Naive Bayes model `nbmodel` using the Train Set with the best `alpha`.

In [ ]:
# Create a Naive Bayes object with the best alpha
nbmodel = MultinomialNB(alpha = 2)

# Train the Naive Bayes model
nbmodel.fit(X_train_counts, y_train["adult"])

#### Goodness of Fit of the Model

Check how good the predictions are on the Train Set.    
Metrics : Classification Accuracy and Confusion Matrix.

In [ ]:
# Import confusion_matrix from sklearn
from sklearn.metrics import confusion_matrix

# Predict Adult corresponding to Text Train
y_train_pred = nbmodel.predict(X_train_counts)

# Print the Classification Accuracy
print("Classification Accuracy \t:", nbmodel.score(X_train_counts, y_train["adult"]))

# Plot the two-way Confusion Matrix
f = plt.figure(figsize=(8, 6))
sb.heatmap(confusion_matrix(y_train, y_train_pred, labels = ["18+", "Not 18+"]),
           annot = True, fmt=".0f", annot_kws={"size": 18},
           xticklabels = ["18+", "Not 18+"], yticklabels = ["18+", "Not 18+"])
plt.title("Train Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")

Check how good the predictions are on the Test Set.    
Metrics : Classification Accuracy and Confusion Matrix.

In [ ]:
# Predict Adult corresponding to Text Test
y_test_pred = nbmodel.predict(X_test_counts)

# Print the Classification Accuracy
print("Classification Accuracy \t:", nbmodel.score(X_test_counts, y_test["adult"]))

# Plot the two-way Confusion Matrix
f = plt.figure(figsize=(8, 6))
sb.heatmap(confusion_matrix(y_test, y_test_pred, labels = ["18+", "Not 18+"]),
           annot = True, fmt=".0f", annot_kws={"size": 18},
           xticklabels = ["18+", "Not 18+"], yticklabels = ["18+", "Not 18+"])
plt.title("Test Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")

Look at Precision, Recall and F1-score for each class on the Test Set.

In [ ]:
# Import classification_report from sklearn
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred))

---

## Does Genre Help?

Train the same model on three different Predictors -- **description only**, **genre only**, and **description + genre** -- using the same Train and Test titles.

In [ ]:
# Import f1_score from sklearn
from sklearn.metrics import f1_score

# Same Train and Test titles, different Predictors
predictors = ["description", "genre", "text"]

for var in predictors:
    vec = CountVectorizer(token_pattern = r"(?u)\b\w+\b")
    train_counts = vec.fit_transform(netflixclean.loc[X_train.index, var])
    test_counts = vec.transform(netflixclean.loc[X_test.index, var])

    model = MultinomialNB(alpha = 2)
    model.fit(train_counts, y_train["adult"])
    pred = model.predict(test_counts)

    print("Predictor :", var)
    print("Classification Accuracy \t:", round(model.score(test_counts, y_test["adult"]), 3))
    print("Macro-F1 \t\t\t:", round(f1_score(y_test["adult"], pred, average = "macro"), 3))
    print()

---

## What the Model Learnt

Compare the probability of each word in the two classes.  
Words with a much higher probability in one class push the prediction towards that class.

In [ ]:
# Words in the vocabulary
words = vectorizer.get_feature_names_out()

# Difference in log-probability : positive means more typical of 18+
adult_idx = list(nbmodel.classes_).index("18+")
other_idx = 1 - adult_idx
diff = pd.Series(nbmodel.feature_log_prob_[adult_idx] - nbmodel.feature_log_prob_[other_idx], index = words)

# Split into Genre tokens and Description words
genre_diff = diff[diff.index.str.startswith("g_")]
word_diff = diff[~diff.index.str.startswith("g_")]

print("Genres pointing to 18+ \t\t:", list(genre_diff.nlargest(10).index))
print("Genres pointing to Not 18+ \t:", list(genre_diff.nsmallest(10).index))
print()
print("Words pointing to 18+ \t\t:", list(word_diff.nlargest(10).index))
print("Words pointing to Not 18+ \t:", list(word_diff.nsmallest(10).index))

---

## Error Analysis

Which of the original ratings are misclassified most often on the Test Set?

In [ ]:
# Summarize the Actuals and Predictions
y_test_pred = pd.DataFrame(y_test_pred, columns = ["PredAdult"], index = y_test.index)
netflix_acc = pd.concat([netflixclean.loc[y_test.index, ["title", "rating", "listed_in", "description"]],
                         y_test, y_test_pred], axis = 1)
netflix_acc["Correct"] = netflix_acc["adult"] == netflix_acc["PredAdult"]

# Accuracy for each original Rating
print(netflix_acc.groupby("rating")["Correct"].mean().sort_values().round(2))

In [ ]:
# A few Misclassified Titles
netflix_acc[netflix_acc["Correct"] == False].head(10)

---

## Prediction for Specific Titles

Let's predict the class for a few specific titles using the model derived above, along with the Class Probabilities.

In [ ]:
# Extract Titles for Prediction
netflix_pred = netflixclean[netflixclean["title"].isin(["Narcos", "Peppa Pig", "Stranger Things"])]

# Predict Response corresponding to Predictors
X_pred = vectorizer.transform(netflix_pred["text"])
y_pred = nbmodel.predict(X_pred)

# Summarize the Actuals and Predictions
y_pred = pd.DataFrame(y_pred, columns = ["PredAdult"], index = netflix_pred.index)
netflix_pred_acc = pd.concat([netflix_pred[["title", "rating", "adult"]], y_pred], axis = 1)
netflix_pred_acc

In [ ]:
# Predict Probabilities corresponding to Predictors
y_prob = nbmodel.predict_proba(X_pred)
np.set_printoptions(precision = 3)
print(nbmodel.classes_)
print(y_prob)